In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q gdown ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.5 MB/s eta 0:00:0000:01


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [4]:
!gdown 1RTJJOtOxiTXwbrMdVlXJ1UMBDnDuFB-E -O custom_syringe_yolo.zip
!ls -la custom_syringe_yolo.zip

Downloading...
From: https://drive.google.com/uc?id=1RTJJOtOxiTXwbrMdVlXJ1UMBDnDuFB-E
To: /kaggle/working/custom_syringe_yolo.zip
100%|██████████████████████████████████████| 17.9M/17.9M [00:00<00:00, 63.4MB/s]
-rw-r--r-- 1 root root 17887866 Aug  5 09:39 custom_syringe_yolo.zip


In [5]:
!unzip -o -q custom_syringe_yolo.zip -d /kaggle/working/
!find /kaggle/working -iname "data.yaml"

/kaggle/working/custom_syringe_yolo/data.yaml


In [6]:
import yaml

data_yaml_path = "/kaggle/working/custom_syringe_yolo/data.yaml"
with open(data_yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/kaggle/working/custom_syringe_yolo"

with open(data_yaml_path, "w") as f:
    yaml.dump(cfg, f)

print(cfg)

{'path': '/kaggle/working/custom_syringe_yolo', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'syringe'}}


In [7]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="/kaggle/working/custom_syringe_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/kaggle/working/runs_syringe",
    name="yolo26n_syringe",
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/custom_syringe_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0

In [8]:
model.export(format="onnx", imgsz=640)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/kaggle/working/runs_syringe/yolo26n_syringe/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 337ms
Prepared 2 packages in 302ms
Installed 2 packages in 15ms
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export succes

'/kaggle/working/runs_syringe/yolo26n_syringe/weights/best.onnx'

In [9]:
import shutil

save_dir = str(results.save_dir)
shutil.make_archive("/kaggle/working/syringe_results", 'zip', save_dir)
print("Archive ready: /kaggle/working/syringe_results.zip")

Archive ready: /kaggle/working/syringe_results.zip
